# 07_icf_trajectory  (oesophagus)

This notebook asks whether the way a patient's ICF scores move before surgery, the trajectory, carries information that a single preoperative value does not. A patient whose function is declining in the weeks before surgery may be a different risk from one who is stably low at the same final value, and only a trajectory can tell them apart.

It does three things:

* checks that patients actually have enough repeated preoperative measurements for a trajectory to be meaningful
* builds two trajectory features per domain, the change from the earliest to the latest preoperative value, and the slope over time toward surgery
* tests whether the trajectory differs between the low and high physiotherapy conclusions, and whether it adds to the surgical preoperative model

Pulmonary complications, major complications, and prolonged stay are the outcomes. Only notes before surgery are used, features are complete case, and a higher ICF score means worse function.

## 1. Setup and load

In [9]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy import stats as st
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
np.random.seed(42)
THESIS=Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
DERIVED=THESIS/"data_derived"; ICF_OUT=THESIS/"outputs"/"icf"; ICF_OUT.mkdir(parents=True, exist_ok=True)
coh=pd.read_csv(DERIVED/"cohort_clean.csv"); notes=pd.read_csv(ICF_OUT/"icf_notes_with_conclusion.csv")
def pad7(s): return pd.Series(s).astype(str).str.strip().str.replace(r'\.0$','',regex=True).str.zfill(7)
coh['MDN']=pad7(coh['upn']); notes['MDN']=pad7(notes['MDN'])
print('cohort', coh.shape, '| notes', notes.shape)

cohort (1022, 57) | notes (504964, 26)


## 2. Preoperative notes for the oesophagus patients

In [10]:
coh['surgery_dt']=pd.to_datetime(coh['surgery_date'],errors='coerce')
notes['note_dt']=pd.to_datetime(notes['NOTITIE_DATUM_dt'],errors='coerce')
sd=coh.set_index('MDN')['surgery_dt']
n=notes[notes['MDN'].isin(set(coh['MDN']))].copy(); n['surgery_dt']=n['MDN'].map(sd)
preop=n[n['note_dt'].notna() & n['surgery_dt'].notna() & (n['note_dt'] < n['surgery_dt'])].copy()
preop['days_to_surgery']=(preop['surgery_dt']-preop['note_dt']).dt.days
# A trajectory needs several measurements spread over time, so unlike the one-month snapshot in 06 we keep
# the whole preoperative period here; tightening to one month would leave too few repeated measurements to
# fit a slope. Set TRAJ_WINDOW_DAYS to a number to bound the look-back; None keeps all preoperative notes.
TRAJ_WINDOW_DAYS=None
if TRAJ_WINDOW_DAYS is not None:
    preop=preop[preop['days_to_surgery']<=TRAJ_WINDOW_DAYS].copy()
KEY=['INS-B455_lvl','ADM-B440_lvl','MBW-B530_lvl','ETN-D550_lvl','ENR-B1300_lvl','FAC-D540_lvl']
KEY=[c for c in KEY if c in notes.columns]
DOMAIN={'INS-B455_lvl':'exercise tolerance','ADM-B440_lvl':'respiration','MBW-B530_lvl':'weight maintenance',
        'ETN-D550_lvl':'eating','ENR-B1300_lvl':'energy','FAC-D540_lvl':'walking'}
print(f"oesophagus patients with preoperative notes: {preop['MDN'].nunique()} | preoperative notes: {len(preop):,}")

oesophagus patients with preoperative notes: 516 | preoperative notes: 23,283


## 3. Viability: are there enough repeated measurements to form a trajectory

A slope or a change is only meaningful if a patient has several preoperative measurements of the same domain. We check that before building anything.

In [11]:
vrows=[]
for c in KEY:
    cnt=preop.dropna(subset=[c]).groupby('MDN').size()
    vrows.append({'domain':DOMAIN.get(c,c),'col':c,'patients_ge1':len(cnt),'patients_ge2':int((cnt>=2).sum()),
                  'patients_ge3':int((cnt>=3).sum()),'patients_ge5':int((cnt>=5).sum()),'median_measurements':int(cnt.median()) if len(cnt) else 0})
viability=pd.DataFrame(vrows)
print(viability.to_string(index=False))
fig,ax=plt.subplots(figsize=(9,4)); x=np.arange(len(viability))
ax.bar(x,viability['patients_ge3'],color='steelblue'); ax.set_xticks(x); ax.set_xticklabels(viability['domain'],rotation=20,ha='right')
ax.set_ylabel('patients with 3 or more preoperative measurements'); ax.set_title('Trajectory viability per ICF domain')
plt.tight_layout(); plt.savefig(ICF_OUT/'trajectory_viability.png',dpi=120); plt.close(fig)

            domain           col  patients_ge1  patients_ge2  patients_ge3  patients_ge5  median_measurements
exercise tolerance  INS-B455_lvl           499           456           402           281                    5
       respiration  ADM-B440_lvl           493           447           399           304                    6
weight maintenance  MBW-B530_lvl           513           507           489           453                   12
            eating  ETN-D550_lvl           514           511           506           459                   14
            energy ENR-B1300_lvl           481           420           369           301                    7
           walking  FAC-D540_lvl           480           441           410           306                    6


## 4. Trajectory features

For each domain and patient we compute two features. The change is the value of the note closest to surgery minus the earliest preoperative value, so a positive change means the score rose, function worsened, as surgery approached. The slope is the linear trend of the score against time toward surgery, per day, fitted only when there are at least three measurements. The change needs at least two.

In [12]:
def traj_feats(df, col):
    out={}
    g=df.dropna(subset=[col])
    for mdn,d in g.groupby('MDN'):
        if len(d)<2: continue
        near=d.loc[d['days_to_surgery'].idxmin(),col]; early=d.loc[d['days_to_surgery'].idxmax(),col]
        rec={col.split('-')[0]+'_change':float(near-early), col.split('-')[0]+'_slope':np.nan}
        if len(d)>=3 and d['days_to_surgery'].nunique()>1:
            x=-d['days_to_surgery'].values.astype(float); y=d[col].values.astype(float)
            rec[col.split('-')[0]+'_slope']=float(np.polyfit(x,y,1)[0])
        out[mdn]=rec
    return pd.DataFrame(out).T
feat=pd.DataFrame(index=sorted(preop['MDN'].unique()))
for c in KEY:
    tf=traj_feats(preop,c)
    if len(tf): feat=feat.join(tf)
feat=feat.reset_index().rename(columns={'index':'MDN'})
concl=preop.dropna(subset=['conc_binary']).groupby('MDN')['conc_binary'].first().rename('conclusion').reset_index()
dat=coh.merge(feat,on='MDN',how='left').merge(concl,on='MDN',how='left')
print('trajectory features built:', [c for c in feat.columns if c!='MDN'])

trajectory features built: ['INS_change', 'INS_slope', 'ADM_change', 'ADM_slope', 'MBW_change', 'MBW_slope', 'ETN_change', 'ETN_slope', 'ENR_change', 'ENR_slope', 'FAC_change', 'FAC_slope']


## 5. Does the trajectory differ between the low and high conclusions

In [13]:
trows=[]
for col in [c for c in dat.columns if c.endswith('_change') or c.endswith('_slope')]:
    hi=dat.loc[dat['conclusion']=='high',col].dropna(); lo=dat.loc[dat['conclusion']=='low',col].dropna()
    if len(hi)>=10 and len(lo)>=10:
        p=st.mannwhitneyu(hi,lo).pvalue
        trows.append({'feature':col,'high_mean':round(hi.mean(),3),'n_high':len(hi),'low_mean':round(lo.mean(),3),'n_low':len(lo),'MWU_p':round(p,3)})
traj_by_concl=pd.DataFrame(trows); traj_by_concl.to_csv(ICF_OUT/'trajectory_by_conclusion.csv',index=False)
print(traj_by_concl.to_string(index=False))

   feature  high_mean  n_high  low_mean  n_low  MWU_p
INS_change     -0.077     191     0.131    203  0.015
 INS_slope      0.000     169     0.002    170  0.102
ADM_change     -0.077     189    -0.081    201  0.882
 ADM_slope     -0.003     167    -0.006    169  0.535
MBW_change      0.069     205    -0.008    226  0.305
 MBW_slope      0.000     201     0.004    210  0.428
ETN_change     -0.719     207    -0.890    228  0.182
 ETN_slope     -0.001     206    -0.003    223  0.342
ENR_change      0.013     175     0.183    187  0.138
 ENR_slope     -0.001     153     0.004    156  0.209
FAC_change     -0.051     191     0.078    191  0.073
 FAC_slope      0.000     179     0.003    168  0.045


## 6. Does the trajectory add to the surgical preoperative model

In [14]:
SURG=['age_at_surgery','sex_male','bmi','asascore','comlong','charlson_cci','anamok_prior_surgery',
      'immunosuppression','neoadj_received','neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno']
SURG=[c for c in SURG if c in dat.columns]
CONT=['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage']
def _cont(cols): return [c for c in cols if c in CONT or c.endswith('_change') or c.endswith('_slope')]
def pipe(cols,C): return Pipeline([('t',ColumnTransformer([('s',StandardScaler(),_cont(cols))],remainder='passthrough')),
                                   ('lr',LogisticRegression(penalty='l2',C=C,solver='liblinear',max_iter=5000,random_state=42))])
def bestC(X,y):
    m=LogisticRegressionCV(Cs=10,cv=5,scoring='neg_log_loss',penalty='l2',solver='liblinear',max_iter=5000,random_state=42)
    Pipeline([('t',ColumnTransformer([('s',StandardScaler(),_cont(list(X.columns)))],remainder='passthrough')),('lr',m)]).fit(X,y); return float(m.C_[0])
def cvauc(X,y):
    y=np.asarray(y); C=bestC(X,y)
    return roc_auc_score(y,cross_val_predict(pipe(list(X.columns),C),X,y,cv=StratifiedKFold(5,shuffle=True,random_state=42),method='predict_proba')[:,1])
rows=[]
for o in ['pulmonary','cd_ge_IIIb','los_prolonged']:
    for f in [c for c in dat.columns if c.endswith('_change') or c.endswith('_slope')]:
        d=dat.dropna(subset=[f,o]+SURG); y=d[o].astype(int)
        if y.nunique()<2 or len(d)<80: continue
        su=cvauc(d[SURG],y); si=cvauc(d[SURG+[f]],y)
        rows.append({'outcome':o,'feature':f,'N':len(d),'events':int(y.sum()),'surgical':round(su,3),'surgical+traj':round(si,3),'gain':round(si-su,3)})
traj_added=pd.DataFrame(rows).sort_values('gain',ascending=False)
traj_added.to_csv(ICF_OUT/'trajectory_added_value.csv',index=False)
print(traj_added.to_string(index=False))

      outcome    feature   N  events  surgical  surgical+traj   gain
los_prolonged  INS_slope 368     219     0.598          0.614  0.016
   cd_ge_IIIb  ETN_slope 466      62     0.597          0.613  0.016
los_prolonged INS_change 424     260     0.561          0.571  0.010
    pulmonary  ADM_slope 355      92     0.562          0.572  0.009
    pulmonary MBW_change 469     115     0.580          0.586  0.006
    pulmonary FAC_change 410     107     0.572          0.576  0.004
    pulmonary ETN_change 472     117     0.586          0.590  0.004
los_prolonged  MBW_slope 449     279     0.579          0.582  0.003
los_prolonged  ENR_slope 342     213     0.590          0.593  0.003
los_prolonged  ETN_slope 466     286     0.593          0.596  0.002
   cd_ge_IIIb  ADM_slope 355      49     0.545          0.546  0.002
    pulmonary  ETN_slope 466     115     0.601          0.602  0.002
   cd_ge_IIIb ADM_change 412      56     0.575          0.575  0.001
   cd_ge_IIIb  MBW_slope 449      

## 7. Figure: added value of the trajectory features

In [15]:
best=traj_added.sort_values('gain',ascending=False).head(12)
fig,ax=plt.subplots(figsize=(11,5)); x=np.arange(len(best))
ax.bar(x,best['gain'],color=['seagreen' if g>0 else 'indianred' for g in best['gain']])
ax.axhline(0,color='black',lw=.8); ax.set_xticks(x); ax.set_xticklabels(best['outcome']+'\n'+best['feature'],rotation=45,ha='right',fontsize=8)
ax.set_ylabel('change in AUC vs surgical model'); ax.set_title('Trajectory features: added value over the surgical model (top 12)')
plt.tight_layout(); plt.savefig(ICF_OUT/'trajectory_added_value.png',dpi=120); plt.close(fig)
print('figures saved to', ICF_OUT)

figures saved to Z:\Users\predicting recovery\AmanDeep\Thesis\outputs\icf


## 8. Reading the results

* Viability first. If patients carry several preoperative measurements per domain, then the trajectory is a fair test rather than a fit across one or two points, and the result can be trusted.
* The trajectory by conclusion tells us whether the high risk patients are the visibly declining ones. If the change and slope do not separate the conclusions, then the conclusion is not capturing a trend that the scores reveal.
* The added value is the central question. If adding a change or a slope to the surgical model does not lift the area under the curve, then the trajectory, like the snapshot before it, carries no information beyond the surgical record.
* Taken with the snapshot analysis in 06, this is the second and final test of the ICF scores. Two views of the same data, tested on a sample rich enough to show a trend, give the same answer, which makes the conclusion of the ICF strand complete: the physiotherapy conclusion and the ICF classifier scores track risk but do not improve prediction beyond a parsimonious preoperative surgical model. That consistency across the surgical variables, the operative course, the assessment, and now the ICF trajectory is the defensible message of the work.

In [16]:
# Preoperative course of ICF functioning (Edwin's request): mean level by weeks before surgery, per key domain.
NAME={'INS-B455_lvl':'Exercise tolerance','ADM-B440_lvl':'Respiration','MBW-B530_lvl':'Weight maintenance',
      'ETN-D550_lvl':'Eating','ENR-B1300_lvl':'Energy and drive','FAC-D540_lvl':'Walking'}
crs=preop.copy()
crs['weeks']=(crs['days_to_surgery']//7)
fig,ax=plt.subplots(figsize=(8,5))
for c in KEY:
    g=crs.dropna(subset=[c]); g=g[g['weeks']<=12].groupby('weeks')[c].mean()
    if len(g)>=2: ax.plot(g.index, g.values, marker='o', ms=3, label=NAME.get(c,c.split('-')[0]))
ax.set_xlabel('Weeks before surgery'); ax.set_ylabel('Mean ICF level')
ax.set_title('Preoperative course of ICF functioning'); ax.invert_xaxis()
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(ICF_OUT/'icf_preoperative_course.png',dpi=300); plt.close(fig)
print('saved icf_preoperative_course.png to', ICF_OUT)


saved icf_preoperative_course.png to Z:\Users\predicting recovery\AmanDeep\Thesis\outputs\icf
